In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Disaster Triage: SMOTE Impact on LightGBM vs XGBoost (ESI 1 vs NOT ESI 1) (`models/train_disaster_triage_3class_exp.ipynb`)

Loads the full emergency cohort directly from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI) using **8 core arrival triage features** and evaluates the impact of **SMOTE (Synthetic Minority Over-sampling Technique)** on **LightGBM** and **XGBoost** for binary resuscitation triage (**`ESI 1` vs `NOT ESI 1`**).

### 🎯 Target Formulation
- **`Class 1: ESI 1 (Immediate Resuscitation)`** ($y=1$): Patients requiring immediate life-saving intervention ($5,271$ visits, $\sim 0.94\%$).
- **`Class 0: NOT ESI 1 (ESI 2–5)`** ($y=0$): Emergent, urgent, and non-urgent visits ($552,758$ visits, $\sim 99.06\%$).

### ⚖️ SMOTE Resampling Protocol
- **Applied strictly to Training Partition**: `SMOTE(random_state=42)` synthetically oversamples minority `ESI 1` cases during model training.
- **Un-resampled Validation & Test Evaluation**: Validation early stopping and holdout test metrics are computed strictly on realistic, un-resampled clinical data.

### 🔬 4 Evaluated Model Configurations
1. **`LightGBM (No SMOTE)`**: Baseline unweighted LightGBM.
2. **`LightGBM (with SMOTE)`**: LightGBM trained on SMOTE-balanced features.
3. **`XGBoost (No SMOTE)`**: Baseline unweighted XGBoost.
4. **`XGBoost (with SMOTE)`**: XGBoost trained on SMOTE-balanced features.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 core triage features + ESI
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c(
  "age", "cc_breathingdifficulty", "gender",
  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"
)

# Export matrices to Python
raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported Full Dataset to Python: %d rows, %d feature columns (all NAs preserved for SimpleImputer)\n", nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve from R, Partition, Median Imputation & SMOTE Resampling
# ---------------------------------------------------------------------------
import os, json, pickle, warnings
from rpy2.robjects import r
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, precision_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender',
    'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
    'triage_vital_rr', 'triage_vital_o2'
]

# Binary Target Formulation: 1 = ESI 1 (Immediate Resuscitation), 0 = NOT ESI 1 (ESI 2-5)
y_all = np.where(esi_all == 1, 1, 0)
LABELS = ['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)']

print("=========================================================")
print("         5v_cleandf DATASET BINARY DISTRIBUTION")
print("=========================================================")
print(f"Total Valid ESI Visits: {len(y_all):,}")
print(f"  * Class 0 [NOT ESI 1 (ESI 2-5)]: {np.sum(y_all == 0):,} ({np.mean(y_all == 0)*100:.2f}%)")
print(f"  * Class 1 [ESI 1 (Resuscitation)]: {np.sum(y_all == 1):,} ({np.mean(y_all == 1)*100:.2f}%)")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print("=========================================================\n")

# Stratified 3-way split: 70% Train, 15% Validation, 15% Holdout Test
itr, itmp = train_test_split(np.arange(len(y_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

raw_tr  = raw_mat_all[itr]
raw_val = raw_mat_all[iva]
raw_te  = raw_mat_all[ite]

y_train = y_all[itr]
y_val   = y_all[iva]
y_test  = y_all[ite]

# Fit SimpleImputer strictly on Training set
print("Fitting SimpleImputer(strategy='median') on Training set...")
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(raw_tr)
X_val   = imputer.transform(raw_val)
X_test  = imputer.transform(raw_te)

print(f"✓ Partition Shapes (Pre-SMOTE): Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

# Apply SMOTE strictly to Training Partition
print("\nApplying SMOTE strictly on Training Partition (random_state=42)...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"✓ SMOTE Resampled Training Shape: {X_train_smote.shape}")
print(f"  * Pre-SMOTE  Class Counts : {{NOT_ESI1: {np.sum(y_train == 0):,}, ESI1: {np.sum(y_train == 1):,}}}")
print(f"  * Post-SMOTE Class Counts : {{NOT_ESI1: {np.sum(y_train_smote == 0):,}, ESI1: {np.sum(y_train_smote == 1):,}}}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Train 4 Models (LightGBM & XGBoost, With and Without SMOTE)
# ---------------------------------------------------------------------------
print("1. Training LightGBM (No SMOTE)...")
lgb_base = LGBMClassifier(
    objective='binary', n_estimators=300, learning_rate=0.05,
    num_leaves=31, max_depth=6, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbosity=-1, n_jobs=-1
)
lgb_base.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(40, verbose=False)])

print("2. Training LightGBM (with SMOTE)...")
lgb_smote = LGBMClassifier(
    objective='binary', n_estimators=300, learning_rate=0.05,
    num_leaves=31, max_depth=6, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbosity=-1, n_jobs=-1
)
lgb_smote.fit(X_train_smote, y_train_smote, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(40, verbose=False)])

print("3. Training XGBoost (No SMOTE)...")
xgb_base = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    eval_metric='logloss', early_stopping_rounds=40, n_jobs=-1
)
xgb_base.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print("4. Training XGBoost (with SMOTE)...")
xgb_smote = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    eval_metric='logloss', early_stopping_rounds=40, n_jobs=-1
)
xgb_smote.fit(X_train_smote, y_train_smote, eval_set=[(X_val, y_val)], verbose=False)

print("\n✓ All 4 models successfully trained!")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Comparative Holdout Test Evaluation Across All 4 Models
# ---------------------------------------------------------------------------
models_dict = {
    'LightGBM (No SMOTE)': lgb_base,
    'LightGBM (with SMOTE)': lgb_smote,
    'XGBoost (No SMOTE)': xgb_base,
    'XGBoost (with SMOTE)': xgb_smote
}

report_rows = []
preds_dict = {}
probs_dict = {}

for name, m in models_dict.items():
    p_test = m.predict_proba(X_test)[:, 1]
    pred   = m.predict(X_test)
    
    probs_dict[name] = p_test
    preds_dict[name] = pred
    
    acc      = accuracy_score(y_test, pred)
    bal_acc  = balanced_accuracy_score(y_test, pred)
    rec_esi1 = recall_score(y_test == 1, pred == 1, zero_division=0)
    spec_not = recall_score(y_test == 0, pred == 0, zero_division=0)
    prec_esi1= precision_score(y_test == 1, pred == 1, zero_division=0)
    f1_esi1  = f1_score(y_test == 1, pred == 1, zero_division=0)
    auc_val  = roc_auc_score(y_test, p_test)
    pr_auc   = average_precision_score(y_test, p_test)
    
    report_rows.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Balanced_Accuracy': round(bal_acc, 4),
        'ESI1_Sensitivity (Recall)': round(rec_esi1, 4),
        'Specificity (NOT ESI 1 Recall)': round(spec_not, 4),
        'ESI1_Precision': round(prec_esi1, 4),
        'ESI1_F1': round(f1_esi1, 4),
        'ROC_AUC': round(auc_val, 4),
        'PR_AUC': round(pr_auc, 4)
    })

report_df = pd.DataFrame(report_rows)
print("=====================================================================================================================")
print("   HOLDOUT TEST COMPARISON: LIGHTGBM vs XGBOOST (WITH AND WITHOUT SMOTE)")
print("=====================================================================================================================")
print(report_df.to_string(index=False))
print("=====================================================================================================================\n")

reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'esi1_binary_smote_lgb_vs_xgb_report.csv')
report_df.to_csv(report_file, index=False)
print(f"Comparative report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: 2x2 Grid of Confusion Matrix Heatmaps (All 4 Model Configurations)
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes = axes.flatten()
colormaps = ['Blues', 'Purples', 'Oranges', 'Greens']

for idx, (name, pred) in enumerate(preds_dict.items()):
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    annot = np.empty_like(cm, dtype=object)
    for i in range(2):
        for j in range(2):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.2f}%)"
            
    sns.heatmap(
        cm_norm,
        annot=annot,
        fmt='',
        cmap=colormaps[idx],
        cbar=True,
        ax=axes[idx],
        vmin=0,
        vmax=1,
        xticklabels=['NOT ESI 1 (2-5)', 'ESI 1 (Resus)'],
        yticklabels=['NOT ESI 1 (2-5)', 'ESI 1 (Resus)']
    )
    
    rec_val = report_rows[idx]['ESI1_Sensitivity (Recall)']
    spec_val = report_rows[idx]['Specificity (NOT ESI 1 Recall)']
    bal_val = report_rows[idx]['Balanced_Accuracy']
    axes[idx].set_title(
        f'{name}\n'
        f'ESI 1 Recall: {rec_val*100:.2f}% | Specificity: {spec_val*100:.2f}% | Bal Acc: {bal_val*100:.2f}%',
        fontsize=11.5,
        fontweight='bold',
        pad=10
    )
    axes[idx].set_xlabel('Predicted Acuity', fontsize=10.5, fontweight='bold')
    axes[idx].set_ylabel('True Acuity', fontsize=10.5, fontweight='bold')

plt.suptitle('SMOTE Impact on ESI 1 Resuscitation Classification (LightGBM vs XGBoost)', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, 'esi1_smote_confusion_matrices.png')
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'esi1_smote_confusion_matrices.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'disaster_triage_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion matrix grid saved to: {cm_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: Comparative ROC and Precision-Recall Curves (All 4 Configurations)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = {
    'LightGBM (No SMOTE)': '#1f77b4',
    'LightGBM (with SMOTE)': '#9467bd',
    'XGBoost (No SMOTE)': '#ff7f0e',
    'XGBoost (with SMOTE)': '#2ca02c'
}

# Panel 1: ROC Curves
for name in models_dict.keys():
    p_test = probs_dict[name]
    fpr, tpr, _ = roc_curve(y_test, p_test)
    auc_score = roc_auc_score(y_test, p_test)
    axes[0].plot(fpr, tpr, color=colors[name], linewidth=2.0, label=f"{name} (AUC = {auc_score:.4f})")

axes[0].plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Chance (0.5000)')
axes[0].set_title('ROC Curves: SMOTE Impact on LightGBM vs XGBoost', fontsize=12.5, fontweight='bold', pad=12)
axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('True Positive Rate (Sensitivity)', fontsize=11, fontweight='bold')
axes[0].legend(loc="lower right", fontsize=10, frameon=True, framealpha=0.95)
axes[0].grid(True, linestyle='--', alpha=0.4)

# Panel 2: Precision-Recall Curves
baseline_pr = np.mean(y_test == 1)
for name in models_dict.keys():
    p_test = probs_dict[name]
    prec, rec, _ = precision_recall_curve(y_test, p_test)
    pr_auc_score = average_precision_score(y_test, p_test)
    axes[1].plot(rec, prec, color=colors[name], linewidth=2.0, label=f"{name} (PR-AUC = {pr_auc_score:.4f})")

axes[1].axhline(y=baseline_pr, color='gray', linestyle='--', linewidth=1.2, label=f'Baseline Proportion ({baseline_pr:.4f})')
axes[1].set_title('Precision-Recall Curves: SMOTE Impact on LightGBM vs XGBoost', fontsize=12.5, fontweight='bold', pad=12)
axes[1].set_xlabel('Recall (Sensitivity)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Precision', fontsize=11, fontweight='bold')
axes[1].legend(loc="upper right", fontsize=10, frameon=True, framealpha=0.95)
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
curves_file = os.path.join(plots_dir, 'esi1_smote_roc_pr_curves.png')
plt.savefig(curves_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"Comparative curves saved to: {curves_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: Export Production Bundle & Deployment Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle_data = {
    'lgb_base': lgb_base,
    'lgb_smote': lgb_smote,
    'xgb_base': xgb_base,
    'xgb_smote': xgb_smote,
    'imputer': imputer,
    'features': FEATURES,
    'labels': LABELS
}

bundle_file = os.path.join(deploy_dir, 'disaster_triage_esi1_binary.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle_data, f)

manifest = dict(
    target='Binary_ESI1_vs_NOT_ESI1',
    comparison_models=['LightGBM (No SMOTE)', 'LightGBM (with SMOTE)', 'XGBoost (No SMOTE)', 'XGBoost (with SMOTE)'],
    dataset='5v_cleandf_RData',
    total_visits=len(y_all),
    n_esi1=int(np.sum(y_all == 1)),
    n_not_esi1=int(np.sum(y_all == 0)),
    labels=LABELS,
    feature_order=FEATURES,
    n_features=len(FEATURES),
    holdout_metrics=report_rows
)

manifest_file = os.path.join(deploy_dir, 'disaster_triage_esi1_binary_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Exported Bundle:   {bundle_file}")
print(f"✓ Exported Manifest: {manifest_file}")